# Logistic Regression — Counterfactual Analysis

This notebook reconstructs the official Logistic Regression counterfactual-analysis pathway from **Steps 12A–12D**.

It begins from the locked baseline and asks:

> What is the smallest set of currently present phishing-driving TF-IDF features whose influence must be reduced for a legitimate False Positive email to cross from phishing prediction to legitimate prediction?

The notebook covers:

1. decision-boundary verification for all 88 official False Positives;
2. minimum feature-level counterfactual generation;
3. aggregation and ranking of recurring counterfactual evidence;
4. mapping numerical counterfactual features back to the original email text.

No model retraining, coefficient modification, TF-IDF refitting, threshold tuning, or synthetic email generation occurs here.

Run `01_dataset_preparation.ipynb` and `02_official_baseline.ipynb` first in the same runtime.


In [ ]:
import re
import numpy as np
import pandas as pd

from scipy.special import expit
from scipy import sparse
from sklearn.metrics import confusion_matrix

THRESHOLD = 0.50
DECISION_BOUNDARY = 0.0
COUNTERFACTUAL_EPSILON = 1e-9

EXPECTED_TP = 4660
EXPECTED_FP = 88
EXPECTED_FN = 48
EXPECTED_TN = 5920
EXPECTED_FEATURES = 10000

print("Counterfactual-analysis configuration loaded.")


## 1. Verify the locked baseline

The fixed Logistic Regression model, fixed train/test split, and locked 10,000-feature TF-IDF representation are reused without modification.


In [ ]:
required = [
    "lr_model", "tfidf_vectorizer",
    "X_train_tfidf", "X_test_tfidf",
    "X_train", "X_test", "y_train", "y_test"
]

missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing required objects: " + ", ".join(missing)
        + "\nRun notebooks 01 and 02 first in the same runtime."
    )

feature_names = tfidf_vectorizer.get_feature_names_out()
coefficients = lr_model.coef_.ravel()
intercept = float(lr_model.intercept_[0])

model_scores = lr_model.decision_function(X_test_tfidf)
model_probabilities = lr_model.predict_proba(X_test_tfidf)[:, 1]
model_predictions = (model_probabilities >= THRESHOLD).astype(int)

cm = confusion_matrix(y_test, model_predictions)

assert np.array_equal(cm, np.array([[4660, 88], [48, 5920]]))
assert len(feature_names) == EXPECTED_FEATURES

print("Official baseline verified.")
print("Intercept:", intercept)
print("Confusion matrix:")
print(cm)


## 2. Step 12A — decision-boundary verification

For Logistic Regression, the linear decision score is:

`intercept + Σ(TF-IDF value × coefficient)`

Step 12A verifies that the manually calculated decision scores and probabilities match the locked classifier before any counterfactual operation.


In [ ]:
manual_scores = (
    np.asarray(X_test_tfidf.dot(coefficients)).ravel()
    + intercept
)
manual_probabilities = expit(manual_scores)

score_max_difference = float(
    np.max(np.abs(manual_scores - model_scores))
)
probability_max_difference = float(
    np.max(np.abs(manual_probabilities - model_probabilities))
)

print("Manual/model score maximum difference:",
      score_max_difference)
print("Reconstructed/model probability maximum difference:",
      probability_max_difference)

assert np.allclose(manual_scores, model_scores)
assert np.allclose(manual_probabilities, model_probabilities)


In [ ]:
y_test_array = np.asarray(y_test)

fp_positions = np.where(
    (y_test_array == 0) & (model_predictions == 1)
)[0]

assert len(fp_positions) == 88

fp_matrix = X_test_tfidf[fp_positions]
fp_probabilities = model_probabilities[fp_positions]
fp_scores = model_scores[fp_positions]

active_counts = np.diff(fp_matrix.indptr)

phase3_fp_baseline_table = pd.DataFrame({
    "FP_Number": np.arange(1, 89),
    "Test_Position": fp_positions,
    "Original_Index": np.asarray(X_test.index)[fp_positions],
    "Original_Probability": fp_probabilities,
    "Original_Decision_Score": fp_scores,
    "Required_Score_Reduction": fp_scores + COUNTERFACTUAL_EPSILON,
    "Active_TFIDF_Features": active_counts
})

print("False Positive probability:")
print(" min :", fp_probabilities.min())
print(" max :", fp_probabilities.max())
print(" mean:", fp_probabilities.mean())

print("\nFalse Positive decision score:")
print(" min :", fp_scores.min())
print(" max :", fp_scores.max())
print(" mean:", fp_scores.mean())

print("\nActive TF-IDF features per False Positive:")
print(" min :", active_counts.min())
print(" max :", active_counts.max())
print(" mean:", active_counts.mean())

display(phase3_fp_baseline_table.head(10))


### Step 12A locked checkpoint

The original experiment reported:

- intercept = **2.1718442046656**
- False Positive probability: minimum **0.503689**, maximum **0.991030**, mean **0.704714**
- False Positive decision score: minimum **0.014756**, maximum **4.704875**, mean **1.078086**
- active TF-IDF features: minimum **4**, maximum **723**, mean **94.25**

Step 12A generated no counterfactual and changed nothing in the model.


In [ ]:
assert np.isclose(intercept, 2.1718442046656, atol=1e-12)
assert round(float(fp_probabilities.min()), 6) == 0.503689
assert round(float(fp_probabilities.max()), 6) == 0.991030
assert round(float(fp_probabilities.mean()), 6) == 0.704714
assert round(float(fp_scores.min()), 6) == 0.014756
assert round(float(fp_scores.max()), 6) == 4.704875
assert round(float(fp_scores.mean()), 6) == 1.078086
assert int(active_counts.min()) == 4
assert int(active_counts.max()) == 723
assert np.isclose(float(active_counts.mean()), 94.25)

print("Step 12A locked checkpoint VERIFIED.")


## 3. Step 12B — minimum feature-level counterfactuals

For each False Positive:

1. identify TF-IDF features actually present in the email;
2. calculate each active feature contribution as `TF-IDF × Logistic Regression coefficient`;
3. keep positive phishing-driving contributions;
4. rank them from largest to smallest;
5. fully reduce earlier selected features;
6. partially reduce only the final boundary-crossing feature when possible;
7. reconstruct the modified vector and verify the prediction with the actual locked model.

Here, **minimum** means the minimum number of active positive-contribution features under this specific additive reduction procedure. It does **not** mean a globally minimum textual edit.

Likewise, an **infeasible** case is infeasible only under this positive-feature-reduction operation.


In [ ]:
positive_coefficient_count = int(np.sum(coefficients > 0))
print("Features with positive LR coefficient:",
      positive_coefficient_count)
assert positive_coefficient_count == 4734


In [ ]:
summary_rows = []
detail_rows = []
modified_vectors = {}

for fp_number, test_position in enumerate(fp_positions, start=1):
    row = X_test_tfidf[test_position].tocsr()
    score = float(model_scores[test_position])
    probability = float(model_probabilities[test_position])
    required = score + COUNTERFACTUAL_EPSILON

    indices = row.indices
    values = row.data
    coeffs = coefficients[indices]
    contributions = values * coeffs

    positive_mask = contributions > 0
    p_indices = indices[positive_mask]
    p_values = values[positive_mask]
    p_coeffs = coeffs[positive_mask]
    p_contrib = contributions[positive_mask]

    order = np.argsort(-p_contrib)
    p_indices = p_indices[order]
    p_values = p_values[order]
    p_coeffs = p_coeffs[order]
    p_contrib = p_contrib[order]

    feasible = float(p_contrib.sum()) > required

    if not feasible:
        summary_rows.append({
            "FP_Number": fp_number,
            "Test_Position": int(test_position),
            "Original_Probability": probability,
            "Original_Decision_Score": score,
            "Required_Score_Reduction": required,
            "Positive_Active_Features": len(p_indices),
            "Counterfactual_Feasible": False,
            "Minimal_Feature_Count": np.nan,
            "Full_Removals": np.nan,
            "Partial_Final_Feature": False,
            "Counterfactual_Probability": np.nan,
            "Counterfactual_Decision_Score": np.nan,
            "Prediction_Flipped": False
        })
        continue

    modified = row.copy().tolil()
    cumulative = 0.0
    selected = []

    for rank, (idx, old_value, coef, contribution) in enumerate(
        zip(p_indices, p_values, p_coeffs, p_contrib), start=1
    ):
        remaining = required - cumulative

        if contribution > remaining:
            reduction_value = remaining / coef
            new_value = max(0.0, old_value - reduction_value)
            achieved = (old_value - new_value) * coef
            fully_removed = False
            boundary_feature = True
        else:
            new_value = 0.0
            achieved = contribution
            fully_removed = True
            boundary_feature = False

        modified[0, idx] = new_value
        cumulative += achieved

        selected.append({
            "Counterfactual_Feature_Rank": rank,
            "Feature": feature_names[idx],
            "TFIDF_Index": int(idx),
            "Original_TFIDF_Value": float(old_value),
            "New_TFIDF_Value": float(new_value),
            "Fraction_Removed": float(
                (old_value - new_value) / old_value
            ),
            "LR_Coefficient": float(coef),
            "Original_Phishing_Contribution": float(contribution),
            "Achieved_Score_Reduction": float(achieved),
            "Fully_Removed": bool(fully_removed),
            "Final_Boundary_Feature": bool(boundary_feature)
        })

        if cumulative >= required:
            break

    modified = modified.tocsr()
    cf_score = float(lr_model.decision_function(modified)[0])
    cf_probability = float(
        lr_model.predict_proba(modified)[0, 1]
    )
    cf_prediction = int(cf_probability >= THRESHOLD)
    flipped = cf_prediction == 0

    modified_vectors[fp_number] = modified

    for item in selected:
        detail_rows.append({
            "FP_Number": fp_number,
            "Test_Position": int(test_position),
            **item
        })

    summary_rows.append({
        "FP_Number": fp_number,
        "Test_Position": int(test_position),
        "Original_Probability": probability,
        "Original_Decision_Score": score,
        "Required_Score_Reduction": required,
        "Positive_Active_Features": len(p_indices),
        "Counterfactual_Feasible": True,
        "Minimal_Feature_Count": len(selected),
        "Full_Removals": sum(x["Fully_Removed"] for x in selected),
        "Partial_Final_Feature":
            any(x["Final_Boundary_Feature"] for x in selected),
        "Counterfactual_Probability": cf_probability,
        "Counterfactual_Decision_Score": cf_score,
        "Prediction_Flipped": flipped
    })

counterfactual_summary = pd.DataFrame(summary_rows)
counterfactual_details = pd.DataFrame(detail_rows)

feasible_summary = counterfactual_summary[
    counterfactual_summary["Counterfactual_Feasible"]
].copy()

print("Total baseline FPs analysed:", len(counterfactual_summary))
print("Feasible:", len(feasible_summary))
print("Infeasible:", 88 - len(feasible_summary))
print("Verified flips:",
      int(feasible_summary["Prediction_Flipped"].sum()))


In [ ]:
assert len(feasible_summary) == 56
assert (88 - len(feasible_summary)) == 32
assert int(feasible_summary["Prediction_Flipped"].sum()) == 56
assert len(counterfactual_details) == 411

sizes = feasible_summary["Minimal_Feature_Count"].astype(int)

print("Minimum:", sizes.min())
print("Maximum:", sizes.max())
print("Mean   :", sizes.mean())
print("Median :", sizes.median())

assert sizes.min() == 1
assert sizes.max() == 61
assert np.isclose(sizes.mean(), 7.339285714285714)
assert sizes.median() == 3.5

size_distribution = (
    sizes.value_counts()
    .sort_index()
    .rename_axis("Minimal_Feature_Count")
    .reset_index(name="FP_Count")
)
size_distribution["Percentage_of_Feasible_FPs"] = (
    100 * size_distribution["FP_Count"] / 56
)

display(size_distribution)


### Example: False Positive case 2

The original experiment found that case 2 crossed the boundary by fully reducing `com` and then partially reducing `net`.

The fractional reduction of `net` is a **numerical TF-IDF reduction**. It does not mean that the same percentage of physical occurrences of the word was deleted from the email.


In [ ]:
fp2 = counterfactual_details[
    counterfactual_details["FP_Number"] == 2
].copy()

display(fp2)

fp2_summary = counterfactual_summary[
    counterfactual_summary["FP_Number"] == 2
].iloc[0]

assert np.isclose(
    fp2_summary["Original_Probability"], 0.57640502,
    atol=1e-8
)
assert np.isclose(
    fp2_summary["Original_Decision_Score"], 0.30803282,
    atol=1e-8
)

assert fp2["Feature"].tolist() == ["com", "net"]

print("FP2 counterfactual example VERIFIED.")


## 4. Step 12C — recurring counterfactual feature evidence

The 411 feature-level counterfactual rows are aggregated across the 56 feasible cases.

The **Counterfactual Evidence Score** is used only for ranking:

`FP frequency × mean achieved score reduction × Logistic Regression coefficient`

It is not a probability and does not modify the classifier.


In [ ]:
grouped = (
    counterfactual_details
    .groupby(["TFIDF_Index", "Feature"], as_index=False)
    .agg(
        CF_FP_Frequency=("FP_Number", "nunique"),
        CF_Total_Occurrences=("FP_Number", "size"),
        Mean_Achieved_Score_Reduction=(
            "Achieved_Score_Reduction", "mean"
        ),
        Total_Achieved_Score_Reduction=(
            "Achieved_Score_Reduction", "sum"
        ),
        LR_Coefficient=("LR_Coefficient", "first"),
        Complete_Removal_Count=("Fully_Removed", "sum"),
        Partial_Reduction_Count=(
            "Fully_Removed", lambda x: int((~x.astype(bool)).sum())
        ),
        Boundary_Feature_Count=("Final_Boundary_Feature", "sum")
    )
)

grouped["Percentage_of_Feasible_FPs"] = (
    100 * grouped["CF_FP_Frequency"] / 56
)

grouped["Counterfactual_Evidence_Score"] = (
    grouped["CF_FP_Frequency"]
    * grouped["Mean_Achieved_Score_Reduction"]
    * grouped["LR_Coefficient"]
)

counterfactual_feature_ranking = grouped.sort_values(
    "Counterfactual_Evidence_Score",
    ascending=False
).reset_index(drop=True)

counterfactual_feature_ranking.insert(
    0, "Counterfactual_Rank",
    np.arange(1, len(counterfactual_feature_ranking) + 1)
)

recurring_features = counterfactual_feature_ranking[
    counterfactual_feature_ranking["CF_FP_Frequency"] >= 3
].copy()

print("Feature-level rows:", len(counterfactual_details))
print("Unique counterfactual features:",
      len(counterfactual_feature_ranking))
print("Recurring features (>=3 FPs):",
      len(recurring_features))

assert len(counterfactual_details) == 411
assert len(counterfactual_feature_ranking) == 268
assert len(recurring_features) == 31

display(counterfactual_feature_ranking.head(30))


The locked recurring ranking begins with:

1. `escapelong escapelong`
2. `com`
3. `net`
4. `great`
5. `online`

`com` occurs most often, but `escapelong escapelong` ranks first because the evidence score combines recurrence, achieved score reduction, and the Logistic Regression coefficient.


In [ ]:
recurring_names = set(recurring_features["Feature"])

def coverage_for_top_n(n):
    selected = set(
        recurring_features.head(n)["Feature"].tolist()
    )
    covered = (
        counterfactual_details[
            counterfactual_details["Feature"].isin(selected)
        ]["FP_Number"].nunique()
    )
    return covered, 100 * covered / 56

coverage_rows = []
for n in [5, 10, 20, 30]:
    covered, percentage = coverage_for_top_n(n)
    coverage_rows.append({
        "Top_N_Recurring_Features": n,
        "Covered_Feasible_FPs": covered,
        "Total_Feasible_FPs": 56,
        "Coverage_Percent": percentage
    })

coverage_table = pd.DataFrame(coverage_rows)
display(coverage_table)

assert coverage_table["Covered_Feasible_FPs"].tolist() == [
    26, 37, 45, 46
]

print("Coverage checkpoint VERIFIED.")


Coverage means that **at least one** selected recurring feature is represented in those counterfactual cases. It does not mean the selected features alone solved that percentage of cases.


## 5. Step 12D — map counterfactual features back to email text

Numerical counterfactual evidence must be connected back to the actual legitimate email text before controlled text variants can be constructed.

For each of the 56 feasible cases, the selected TF-IDF feature is searched in the original cleaned email. A separate record is created for each textual occurrence. Context extends up to **35 words on each side** of the matched occurrence.

Step 12D creates structured reconstruction templates only. It does **not** generate synthetic email text.


In [ ]:
def feature_regex(feature):
    # Match the TF-IDF n-gram as consecutive word tokens.
    tokens = feature.split()
    return re.compile(
        r"(?<!\w)" + r"\s+".join(
            re.escape(t) for t in tokens
        ) + r"(?!\w)",
        flags=re.IGNORECASE
    )

def word_context(text, start, end, window=35):
    # Determine word spans in the complete cleaned email.
    words = list(re.finditer(r"\S+", text))
    if not words:
        return ""

    hit_indices = [
        i for i, w in enumerate(words)
        if not (w.end() <= start or w.start() >= end)
    ]
    if not hit_indices:
        return ""

    left = max(0, hit_indices[0] - window)
    right = min(len(words), hit_indices[-1] + window + 1)
    return " ".join(w.group(0) for w in words[left:right])

test_text_array = np.asarray(X_test)

mapping_rows = []
template_rows = []

for _, summary_row in feasible_summary.iterrows():
    fp_number = int(summary_row["FP_Number"])
    test_position = int(summary_row["Test_Position"])
    text = str(test_text_array[test_position])

    details = counterfactual_details[
        counterfactual_details["FP_Number"] == fp_number
    ].sort_values("Counterfactual_Feature_Rank")

    matched_feature_count = 0
    unmatched_feature_count = 0

    for _, d in details.iterrows():
        feature = d["Feature"]
        matches = list(feature_regex(feature).finditer(text))

        if matches:
            matched_feature_count += 1
            for occurrence_number, match in enumerate(matches, start=1):
                mapping_rows.append({
                    "FP_Number": fp_number,
                    "Test_Position": test_position,
                    "Feature": feature,
                    "Counterfactual_Feature_Rank":
                        int(d["Counterfactual_Feature_Rank"]),
                    "Occurrence_Number": occurrence_number,
                    "Matched": True,
                    "Match_Start": match.start(),
                    "Match_End": match.end(),
                    "Context_35_Words_Each_Side":
                        word_context(
                            text, match.start(), match.end(), 35
                        )
                })
        else:
            unmatched_feature_count += 1
            mapping_rows.append({
                "FP_Number": fp_number,
                "Test_Position": test_position,
                "Feature": feature,
                "Counterfactual_Feature_Rank":
                    int(d["Counterfactual_Feature_Rank"]),
                "Occurrence_Number": 0,
                "Matched": False,
                "Match_Start": np.nan,
                "Match_End": np.nan,
                "Context_35_Words_Each_Side": ""
            })

    minimal_features = " | ".join(details["Feature"].tolist())
    recurring_cf = " | ".join(
        [f for f in details["Feature"].tolist()
         if f in recurring_names]
    )

    template_rows.append({
        "FP_Number": fp_number,
        "Test_Position": test_position,
        "Original_Probability":
            float(summary_row["Original_Probability"]),
        "Minimal_Feature_Count":
            int(summary_row["Minimal_Feature_Count"]),
        "Minimal_Counterfactual_Features": minimal_features,
        "Recurring_CF_Features": recurring_cf,
        "Matched_Feature_Count": matched_feature_count,
        "Unmatched_Feature_Count": unmatched_feature_count,
        "Original_Text": text
    })

mapping_table = pd.DataFrame(mapping_rows)
counterfactual_templates = pd.DataFrame(template_rows)

matched_occurrence_rows = int(mapping_table["Matched"].sum())
unmatched_feature_rows = int((~mapping_table["Matched"]).sum())
fully_mapped_templates = int(
    (counterfactual_templates["Unmatched_Feature_Count"] == 0).sum()
)
partially_unmatched_templates = int(
    (counterfactual_templates["Unmatched_Feature_Count"] > 0).sum()
)

print("Matched feature-occurrence rows:",
      matched_occurrence_rows)
print("Unmatched feature rows:",
      unmatched_feature_rows)
print("Fully mapped templates:",
      fully_mapped_templates)
print("Templates with >=1 unmatched feature:",
      partially_unmatched_templates)

display(counterfactual_templates.head(10))


### Step 12D locked checkpoint

The original experiment produced:

- **1,340** matched feature-occurrence rows;
- **5** unmatched feature rows;
- **52** templates with every counterfactual feature mapped;
- **4** templates with at least one unmatched feature.

There are more matched occurrence rows than the 411 feature-level counterfactual rows because a selected feature can occur multiple times in one email.


In [ ]:
# Exact text-token mapping can be affected if the text object passed from
# notebook 01 is not the same cleaned representation used in the original
# Step 12D. These assertions intentionally protect the locked experiment.
assert matched_occurrence_rows == 1340, (
    "Step 12D mapping count differs from the locked experiment. "
    "Confirm that X_test contains the exact cleaned text representation "
    "from notebook 01."
)
assert unmatched_feature_rows == 5
assert fully_mapped_templates == 52
assert partially_unmatched_templates == 4

print("Step 12D locked checkpoint VERIFIED.")


## 6. False Positive case 2 mapping example

For case 2:

- `com` was fully reduced numerically;
- `net` was the final boundary feature and was reduced by approximately **26.25% in TF-IDF space**;
- both features mapped successfully to the original email text.

The 26.25% value is **not** a claim that 26.25% of physical `net` word occurrences were removed.


In [ ]:
fp2_mapping_details = counterfactual_details[
    counterfactual_details["FP_Number"] == 2
][[
    "Counterfactual_Feature_Rank",
    "Feature",
    "Fraction_Removed",
    "Fully_Removed",
    "Final_Boundary_Feature"
]].copy()

fp2_mapping_details["Recurring_Feature"] = (
    fp2_mapping_details["Feature"].isin(recurring_names)
)

fp2_mapping_details["Text_Match"] = (
    fp2_mapping_details["Feature"].isin(
        mapping_table[
            (mapping_table["FP_Number"] == 2)
            & (mapping_table["Matched"])
        ]["Feature"]
    )
)

display(fp2_mapping_details)

assert fp2_mapping_details["Feature"].tolist() == ["com", "net"]
assert np.isclose(
    fp2_mapping_details.iloc[0]["Fraction_Removed"], 1.0
)
assert np.isclose(
    fp2_mapping_details.iloc[1]["Fraction_Removed"],
    0.262539,
    atol=1e-6
)

print("FP2 feature-to-text mapping example VERIFIED.")


## 7. Save and display counterfactual evidence tables

This section exports the nine evidence tables and **displays the key results directly in the notebook**. The complete case-level records are retained for reproducibility, but original email text and surrounding text snippets are excluded from the displayed and public-facing previews. Do not publish the full mapping and template CSV files without checking their contents for private information.

The tables below are generated from the locked model and previously computed counterfactual analysis. No model retraining occurs here.


In [ ]:
OUTPUT_FOLDER = "counterfactual_analysis_outputs"

import os
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

phase3_fp_baseline_table.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12A_Official_88_FP_Counterfactual_Baseline.csv"
    ),
    index=False
)

counterfactual_summary.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12B_Counterfactual_Summary.csv"
    ),
    index=False
)

counterfactual_details.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12B_Counterfactual_Feature_Details.csv"
    ),
    index=False
)

size_distribution.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12B_Minimal_Size_Distribution.csv"
    ),
    index=False
)

counterfactual_feature_ranking.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12C_Counterfactual_Feature_Ranking.csv"
    ),
    index=False
)

recurring_features.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12C_Recurring_Features.csv"
    ),
    index=False
)

coverage_table.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12C_Coverage.csv"
    ),
    index=False
)

mapping_table.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12D_Feature_Text_Mapping.csv"
    ),
    index=False
)

counterfactual_templates.to_csv(
    os.path.join(
        OUTPUT_FOLDER,
        "Step12D_Counterfactual_Text_Templates.csv"
    ),
    index=False
)

print("Counterfactual evidence saved to:", OUTPUT_FOLDER)

# ------------------------------------------------------------
# Section 7: display reproducible evidence tables in notebook
# ------------------------------------------------------------
from IPython.display import display

public_previews = [
    ("Step 12A — False Positive baseline (first 10)", phase3_fp_baseline_table, 10),
    ("Step 12B — Counterfactual summary (first 10)", counterfactual_summary, 10),
    ("Step 12B — Minimal feature-count distribution", size_distribution, 30),
    ("Step 12C — Top 20 ranked counterfactual features", counterfactual_feature_ranking, 20),
    ("Step 12C — Recurring features (first 20)", recurring_features, 20),
    ("Step 12C — Feature coverage", coverage_table, 20),
]

for title, frame, max_rows in public_previews:
    print("\n" + "=" * 65)
    print(title)
    print(f"Total rows: {len(frame)} | Columns: {len(frame.columns)}")
    display(frame.head(max_rows).reset_index(drop=True))

# Only show aggregated mapping statistics, never email text.
mapping_summary = pd.DataFrame([
    {"Measure": "Official false positives", "Count": len(fp_positions)},
    {"Measure": "Feasible feature-level counterfactuals", "Count": len(feasible_summary)},
    {"Measure": "Feature-level counterfactual rows", "Count": len(counterfactual_details)},
    {"Measure": "Matched text occurrences", "Count": matched_occurrence_rows},
    {"Measure": "Unmatched feature rows", "Count": unmatched_feature_rows},
    {"Measure": "Fully mapped text templates", "Count": fully_mapped_templates},
    {"Measure": "Templates with unmatched features", "Count": partially_unmatched_templates},
])
print("\nStep 12D — Text mapping summary (no email text)")
display(mapping_summary)

# Confirm exported files exist and match in-memory row counts.
exported_tables = [
    ("Step12A_Official_88_FP_Counterfactual_Baseline.csv", phase3_fp_baseline_table),
    ("Step12B_Counterfactual_Summary.csv", counterfactual_summary),
    ("Step12B_Counterfactual_Feature_Details.csv", counterfactual_details),
    ("Step12B_Minimal_Size_Distribution.csv", size_distribution),
    ("Step12C_Counterfactual_Feature_Ranking.csv", counterfactual_feature_ranking),
    ("Step12C_Recurring_Features.csv", recurring_features),
    ("Step12C_Coverage.csv", coverage_table),
    ("Step12D_Feature_Text_Mapping.csv", mapping_table),
    ("Step12D_Counterfactual_Text_Templates.csv", counterfactual_templates),
]

verification_rows = []
for filename, original in exported_tables:
    path = os.path.join(OUTPUT_FOLDER, filename)
    exists = os.path.isfile(path)
    saved_rows = len(pd.read_csv(path)) if exists else None
    verified = exists and saved_rows == len(original)
    verification_rows.append({
        "File": filename,
        "Expected_Rows": len(original),
        "Saved_Rows": saved_rows,
        "Verified": verified,
    })

verification_table = pd.DataFrame(verification_rows)
print("\nSection 7 — CSV export verification")
display(verification_table)
assert verification_table["Verified"].all(), "An exported evidence table failed verification."
print("All nine evidence tables verified. No model retraining performed.")


## Counterfactual-analysis checkpoint

The locked pathway reproduced in this notebook is:

**88 official False Positives → 56 feasible feature-level counterfactuals → 411 counterfactual feature rows → 268 unique features → 31 recurring features → 52 fully mapped text templates**

The other 32 False Positives were infeasible **under the specified positive-feature-reduction operation**, not necessarily under every possible counterfactual method.

No synthetic legitimate email has been generated yet.

The next core notebook is:

`05_synthetic_variant_generation.ipynb`
